In [1]:
# Import the libraries
from qiskit import QuantumCircuit
from qiskit.circuit.library import QFT
from qiskit.quantum_info import Statevector
import numpy as np
# ------------------------------------------------------------------- --------------------------------------------------------------
# Uncomment this part of code if you need to set up the evolution of a free-particle wave packet test problem
# ----------------------------------------------------------------- ----------------------------------------------------------------
# Constants
n = 8
M = 2**n
r_min = 0
r_max = 5
dr = (r_max-r_min)/M
mu = 0.9412*1822.888
r_s = 1
a = 0.25
p_s = 30
N = 100
t_0 = 0
t_fin = 150
dt = (t_fin - t_0)/N
# Define the spatial grid
r = np.array([r_min + (m+0.5)*dr for m in range(M)])
def initial_wave_packet(qc, n):
# Initializes a Gaussian wave packet
    gaussian_wave_packet = np.zeros(M, dtype=complex)
    for m in range(M):
        gaussian_wave_packet[m] = (2/np.pi)**0.25 * np.sqrt(1/a) * np.exp(-((r[m] - r_s) / a)**2 + 1j*(r[m] - r_s)*p_s)
        summ = np.sum(np.abs(gaussian_wave_packet) ** 2)
        gaussian_wave_packet /= np.sqrt(summ)
        qc.initialize(gaussian_wave_packet, range(n))
        return qc
def potential_energy_operator(qc, scale_factor=1):
# Potential energy operator: V = 0 (for free particle)
    return qc
# -----------------------------------------------------------------------------------------------------------------------------------
# # ---------------------------------------------------------------------------------------------------------------------------------
# # Uncomment this part of code if you need to set up the quantum tunneling through the barrier test problem
# # ---------------------------------------------------------------------------------------------------------------------------------
# # Constants
n = 7
M = 2**n
r_min = 0
r_max = 5
dr = (r_max-r_min)/M
mu = 0.9412*1822.888
N = 100
t_0 = 0
t_fin = 300
dt = (t_fin - t_0)/N
V_min = -0.017
# Define the spatial grid
r = np.array([r_min + (m+0.5)*dr for m in range(M)])
def initial_wave_packet(qc, n):
# Initializes a uniform wave packet
    for j in range(n-2):
        qc.ry(np.pi/2, j)
        qc.ry(np.pi, n-2)
    return qc
def potential_energy_operator(qc, scale_factor=1):
# Potential energy operator: V = double-well potential
    qc.p(-V_min*dt*scale_factor, n-2)
    return qc
# # ---------------------------------------------------------------------------------------------------------------------------------
# # ---------------------------------------------------------------------------------------------------------------------------------
# # Uncomment this part of code if you need to set up the vibrations of harmonic oscillator test problem
# # ---------------------------------------------------------------------------------------------------------------------------------
# Constants
n = 8
M = 2**n
r_min = 0
r_max = 5
dr = (r_max-r_min)/M
mu = 0.9412*1822.888
r_s = 1.5
a = 0.36
p_s = 0
N = 100
t_0 = 0
t_fin = 1100
dt = (t_fin - t_0)/N
r_eq = 2.5
H_to_wn = 219474.63
omega = 3978.6 / H_to_wn
# Define the spatial grid
r = np.array([r_min + (m+0.5)*dr for m in range(M)])
def initial_wave_packet(qc, n):
# Initializes a Gaussian wave packet
    gaussian_wave_packet = np.zeros(M, dtype=complex)
    for m in range(M):
        gaussian_wave_packet[m] = (2 / np.pi) ** 0.25 * np.sqrt(1/a) * np.exp(-((r[m] - r_s) / a) ** 2 + 1j*(r[m] - r_s)*p_s)
        summ = np.sum(np.abs(gaussian_wave_packet) ** 2)
        gaussian_wave_packet /= np.sqrt(summ)
        qc.initialize(gaussian_wave_packet, range(n))
    return qc
def potential_energy_operator(qc, scale_factor=1):
# Potential energy operator: V = harmonic oscillator potential
    k = mu * omega**2
    alpha = -k*dr**2*dt/2 *scale_factor
    beta = alpha*(2*r_min-2*r_eq+dr)/dr
    gamma = alpha * ((2*r_min-2*r_eq+dr)/(2*dr))**2
# Quadratic term
    for j in np.arange(n):
        qc.p((alpha*2**(2*j)),j)
    for j in np.arange(n-1):
        for k in np.arange(j+1,n,1):
            qc.cp((2*alpha*2**(j+k)),j,k)
# Linear term
    for j in np.arange(n):
        qc.p((beta*2**j),j)
    return qc
# # ------------------------------------------------------------------------------------------------------------------------
# --------------------------------------------------------------------------------------------------------------------------
# Starting from here, the code is general, for any potential energy function
# --------------------------------------------------------------------------------------------------------------------------
# Quantum circuit for kinetic energy operator
def kinetic_energy_operator(qc):
# Define QFT and IQFT: Change to approximation_degree=1 to use approximate QFT or IQFT
    qft = QFT(num_qubits=n, approximation_degree=0, do_swaps=False, inverse=False, insert_barriers=True, name=None)
    iqft = QFT(num_qubits=n, approximation_degree=0, do_swaps=False, inverse=True, insert_barriers=True, name=None)
# Kinetic energy operator starts here
    qc.append(qft, range(n))
    qc.x(0)
    theta = -(2*np.pi/(r_max-r_min))**2 * dt/(2*mu)
    phi = -theta*M
    delta = theta*M**2/4
# Quadratic term
    for j in np.arange(n):
        qc.p((theta*2**(2*j)),n-1-j)
    for j in np.arange(n-1):
        for k in np.arange(j+1,n,1):
            qc.cp((2*theta*2**(j+k)),n-1-j,n-1-k)
# Linear term
    for j in np.arange(n):
        qc.p((phi*2**j), n-1-j)
        qc.x(0)
        qc.append(iqft, range(n))
    return qc
# Simulation loop to calculate |ψ(r_m, t)|^2
psi_squared = np.zeros((M, N+1))
for t in range(N+1):
# Initialize quantum circuit with n qubits
    qc = QuantumCircuit(n)
initial_wave_packet(qc, n)
# Time evolution loop
for time_step in range(t):
    potential_energy_operator(qc,scale_factor=0.5) # Apply half potential energy operator
    kinetic_energy_operator(qc) # Apply kinetic energy operator
    potential_energy_operator(qc,scale_factor=0.5) # Apply half potential energy operator
# Compute probabilities from the quantum statevector
probabilities = {format(m, f"0{n}b"): 0 for m in range(M)}
v = Statevector.from_instruction(qc)
probabilities.update(v.probabilities_dict())
probabilities = {r[m]: probabilities[format(m, f"0{n}b")] for m in range(M)}
# Store |ψ(r_m, t)|^2 values
psi_squared[:, t] = list(probabilities.values())
# Save probability densities to a text file
with open("psi_squared_classical_emulator.txt", "w") as file:
    for t in range(N+1):
        for m in range(M):
            file.write(f"{psi_squared[m,t]}\n")

/tmp/ipykernel_872475/4158585713.py:120: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  qft = QFT(num_qubits=n, approximation_degree=0, do_swaps=False, inverse=False, insert_barriers=True, name=None)
/tmp/ipykernel_872475/4158585713.py:121: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  iqft = QFT(num_qubits=n, approximation_degree=0, do_swaps=False, inverse=True, insert_barriers=True, name=None)
